In [1]:
"""
ISIC 2019 Dataset: Clean Metadata Alignment with Existing Train/Val/Test Splits
This script aligns metadata CSV with your pre-split image folders and creates
a clean dataset in /kaggle/working/ for multimodal training.

INPUT STRUCTURE (your existing dataset):
    /kaggle/input/datasets/msalmankhan231103/isic201970/isic2019_split/
    ├── train/
    │   ├── AK/
    │   ├── BCC/
    │   └── ...
    ├── val/
    │   ├── AK/
    │   └── ...
    └── test/
        ├── AK/
        └── ...

METADATA CSV (from ISIC 2019, you provide path):
    Should have columns: image (filename), age_approx, sex, localization, diagnosis, etc.

OUTPUT STRUCTURE (clean copy in working directory):
    /kaggle/working/isic2019_multimodal/
    ├── train/
    │   ├── images/ (copied or symlinked)
    │   └── metadata.csv
    ├── val/
    │   ├── images/
    │   └── metadata.csv
    └── test/
        ├── images/
        └── metadata.csv
"""

import os
import shutil
import pandas as pd
import numpy as np
from pathlib import Path
import json
import warnings
from tqdm import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

warnings.filterwarnings('ignore')

# ============================================
# CONFIGURATION - MODIFY THESE PATHS
# ============================================

# Path to your existing split dataset (input)
INPUT_ROOT = "/kaggle/input/datasets/msalmankhan231103/isic201970/isic2019_split"
TRAIN_INPUT = os.path.join(INPUT_ROOT, "train")
VAL_INPUT = os.path.join(INPUT_ROOT, "val")
TEST_INPUT = os.path.join(INPUT_ROOT, "test")

# Path to your downloaded metadata CSV file
# You can download from: https://www.kaggle.com/datasets/andrewmvd/isic-2019
# Or from: https://challenge.isic-archive.com/data/
METADATA_CSV = "/kaggle/input/datasets/malmank/metadataisic/ISIC_2019_Training_Metadata (2).csv"  # <-- UPDATE THIS

# Output directory for clean multimodal dataset
OUTPUT_ROOT = "/kaggle/working/isic2019_multimodal"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# ============================================
# STEP 1: LOAD AND PREPARE METADATA
# ============================================

print("="*60)
print("STEP 1: Loading Metadata CSV")
print("="*60)

# Load metadata
if not os.path.exists(METADATA_CSV):
    raise FileNotFoundError(f"Metadata file not found at {METADATA_CSV}\n"
                            f"Please upload the ISIC 2019 metadata CSV to that path.")

metadata_df = pd.read_csv(METADATA_CSV)
print(f"Loaded {len(metadata_df)} rows from metadata CSV")
print(f"Columns: {metadata_df.columns.tolist()}")

# Identify the column containing image filenames (usually 'image' or 'isic_id')
image_col = None
for col in ['image', 'isic_id', 'image_name', 'filename']:
    if col in metadata_df.columns:
        image_col = col
        break

if image_col is None:
    raise ValueError("Could not find image filename column in metadata. "
                     f"Available columns: {metadata_df.columns.tolist()}")

# Extract base ISIC ID (remove .jpg/.png extension)
metadata_df['image_id'] = metadata_df[image_col].apply(lambda x: Path(str(x)).stem)
print(f"Extracted {len(metadata_df['image_id'].unique())} unique image IDs")

# Display sample
print("\nMetadata sample:")
print(metadata_df.head(2))

# ============================================
# STEP 2: SCAN IMAGES FROM EACH SPLIT
# ============================================

def scan_split(split_path, split_name):
    """Scan a split folder and return list of image info"""
    images = []
    if not os.path.exists(split_path):
        print(f"Warning: {split_path} does not exist")
        return images
    
    for class_name in os.listdir(split_path):
        class_dir = os.path.join(split_path, class_name)
        if not os.path.isdir(class_dir):
            continue
        for img_file in os.listdir(class_dir):
            if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_id = Path(img_file).stem
                images.append({
                    'image_id': image_id,
                    'filename': img_file,
                    'class': class_name,
                    'split': split_name,
                    'source_path': os.path.join(class_dir, img_file)
                })
    return images

print("\n" + "="*60)
print("STEP 2: Scanning Existing Split Folders")
print("="*60)

train_images = scan_split(TRAIN_INPUT, 'train')
val_images = scan_split(VAL_INPUT, 'val')
test_images = scan_split(TEST_INPUT, 'test')

print(f"Train: {len(train_images)} images")
print(f"Val:   {len(val_images)} images")
print(f"Test:  {len(test_images)} images")

# ============================================
# STEP 3: ALIGN METADATA WITH EACH SPLIT
# ============================================

def align_metadata_for_split(images, metadata_df, split_name):
    """Align metadata to images in a split, fill missing values"""
    
    # Create dataframe from images
    img_df = pd.DataFrame(images)
    
    # Merge with metadata
    merged = img_df.merge(metadata_df, on='image_id', how='left')
    
    # Check for missing metadata
    missing_count = merged['image_id'].isna().sum() if 'image_id' in merged.columns else 0
    if missing_count > 0:
        print(f"  Warning: {missing_count} images in {split_name} have no metadata match")
    
    # Fill missing metadata with defaults
    # Age: use median of available ages
    if 'age_approx' in merged.columns:
        median_age = merged['age_approx'].median() if merged['age_approx'].notna().any() else 50
        merged['age_approx'] = merged['age_approx'].fillna(median_age)
    else:
        merged['age_approx'] = 50
    
    # Sex: default 'unknown'
    if 'sex' in merged.columns:
        merged['sex'] = merged['sex'].fillna('unknown')
    else:
        merged['sex'] = 'unknown'
    
    # Localization: default 'unknown'
    if 'localization' in merged.columns:
        merged['localization'] = merged['localization'].fillna('unknown')
    else:
        merged['localization'] = 'unknown'
    
    # Diagnosis: use class from folder
    if 'diagnosis' in merged.columns:
        merged['diagnosis_meta'] = merged['diagnosis']
    else:
        merged['diagnosis_meta'] = merged['class']
    
    return merged

print("\n" + "="*60)
print("STEP 3: Aligning Metadata with Each Split")
print("="*60)

train_meta = align_metadata_for_split(train_images, metadata_df, 'train')
val_meta = align_metadata_for_split(val_images, metadata_df, 'val')
test_meta = align_metadata_for_split(test_images, metadata_df, 'test')

print(f"Train metadata: {len(train_meta)} rows")
print(f"Val metadata:   {len(val_meta)} rows")
print(f"Test metadata:  {len(test_meta)} rows")

# ============================================
# STEP 4: CREATE CLEAN DATASET IN WORKING DIRECTORY
# ============================================

def create_clean_split(meta_df, split_name, output_root):
    """Create a clean split folder with images (symlinks) and metadata CSV"""
    
    split_dir = os.path.join(output_root, split_name)
    images_dir = os.path.join(split_dir, 'images')
    os.makedirs(images_dir, exist_ok=True)
    
    # Save metadata CSV
    meta_path = os.path.join(split_dir, 'metadata.csv')
    meta_df.to_csv(meta_path, index=False)
    print(f"  Saved metadata to {meta_path}")
    
    # Copy or symlink images
    print(f"  Copying images to {images_dir}...")
    for idx, row in tqdm(meta_df.iterrows(), total=len(meta_df), desc=f"  Copying {split_name}"):
        src_path = row['source_path']
        dst_path = os.path.join(images_dir, row['filename'])
        
        # Use symlink to save space (or copy if symlink not allowed)
        if not os.path.exists(dst_path):
            try:
                os.symlink(src_path, dst_path)
            except:
                shutil.copy2(src_path, dst_path)
    
    return split_dir

print("\n" + "="*60)
print("STEP 4: Creating Clean Dataset in Working Directory")
print("="*60)

# Create train split
create_clean_split(train_meta, 'train', OUTPUT_ROOT)

# Create val split
create_clean_split(val_meta, 'val', OUTPUT_ROOT)

# Create test split
create_clean_split(test_meta, 'test', OUTPUT_ROOT)

# Save class mapping
class_names = sorted(train_meta['class'].unique())
class_to_idx = {name: i for i, name in enumerate(class_names)}
with open(os.path.join(OUTPUT_ROOT, 'class_mapping.json'), 'w') as f:
    json.dump(class_to_idx, f)

print(f"\n✅ Clean dataset created at: {OUTPUT_ROOT}")
print(f"Class mapping: {class_to_idx}")

# ============================================
# STEP 5: VERIFY AND DISPLAY STATISTICS
# ============================================

print("\n" + "="*60)
print("DATASET STATISTICS")
print("="*60)

def print_split_stats(meta_df, split_name):
    print(f"\n{split_name.upper()} SPLIT:")
    print(f"  Total images: {len(meta_df)}")
    print(f"  Classes: {meta_df['class'].nunique()}")
    print(f"  Class distribution:")
    class_counts = meta_df['class'].value_counts()
    for cls, cnt in class_counts.items():
        print(f"    {cls}: {cnt}")
    print(f"  Metadata completeness:")
    for col in ['age_approx', 'sex', 'localization']:
        if col in meta_df.columns:
            non_null = meta_df[col].notna().sum()
            print(f"    {col}: {non_null}/{len(meta_df)} ({100*non_null/len(meta_df):.1f}%)")

print_split_stats(train_meta, 'train')
print_split_stats(val_meta, 'val')
print_split_stats(test_meta, 'test')

# ============================================
# STEP 6: CREATE PYTORCH DATASET CLASS (OPTIONAL, FOR LATER USE)
# ============================================

class MultimodalISICDataset(Dataset):
    """PyTorch Dataset that loads image + metadata"""
    
    def __init__(self, split_dir, transform=None, metadata_cols=None):
        self.split_dir = split_dir
        self.metadata_df = pd.read_csv(os.path.join(split_dir, 'metadata.csv'))
        self.images_dir = os.path.join(split_dir, 'images')
        self.transform = transform
        
        # Default metadata columns
        if metadata_cols is None:
            metadata_cols = ['age_approx', 'sex', 'localization']
        self.metadata_cols = metadata_cols
        
        # Encode categorical columns
        self.sex_map = {'male': 0, 'female': 1, 'unknown': 0.5}
        self.loc_map = {'head/neck': 0, 'trunk': 1, 'upper extremity': 2, 
                        'lower extremity': 3, 'unknown': 4}
        
        # Normalize age
        ages = self.metadata_df['age_approx'].values
        self.age_mean = ages.mean()
        self.age_std = ages.std() if ages.std() > 0 else 1
        
    def __len__(self):
        return len(self.metadata_df)
    
    def _encode_metadata(self, row):
        """Convert metadata row to tensor"""
        # Age (normalized)
        age_norm = (row['age_approx'] - self.age_mean) / self.age_std
        
        # Sex (0/1/0.5)
        sex = self.sex_map.get(row['sex'], 0.5)
        
        # Location (one-hot 5-dim)
        loc_idx = self.loc_map.get(row['localization'], 4)
        loc_onehot = np.zeros(5)
        loc_onehot[loc_idx] = 1
        
        return np.concatenate([[age_norm, sex], loc_onehot])
    
    def __getitem__(self, idx):
        row = self.metadata_df.iloc[idx]
        
        # Load image
        img_path = os.path.join(self.images_dir, row['filename'])
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        # Load metadata
        metadata = self._encode_metadata(row)
        metadata = torch.tensor(metadata, dtype=torch.float32)
        
        # Label
        label = class_to_idx[row['class']]
        
        return image, metadata, label

# Save the dataset class for later use
with open(os.path.join(OUTPUT_ROOT, 'dataset_class.py'), 'w') as f:
    f.write("""
import torch
from torch.utils.data import Dataset
from PIL import Image
import pandas as pd
import numpy as np

class MultimodalISICDataset(Dataset):
    def __init__(self, split_dir, transform=None):
        self.split_dir = split_dir
        self.metadata_df = pd.read_csv(f'{split_dir}/metadata.csv')
        self.images_dir = f'{split_dir}/images'
        self.transform = transform
        self.sex_map = {'male': 0, 'female': 1, 'unknown': 0.5}
        self.loc_map = {'head/neck': 0, 'trunk': 1, 'upper extremity': 2, 
                        'lower extremity': 3, 'unknown': 4}
        ages = self.metadata_df['age_approx'].values
        self.age_mean = ages.mean()
        self.age_std = ages.std() if ages.std() > 0 else 1
    
    def __len__(self):
        return len(self.metadata_df)
    
    def _encode_metadata(self, row):
        age_norm = (row['age_approx'] - self.age_mean) / self.age_std
        sex = self.sex_map.get(row['sex'], 0.5)
        loc_idx = self.loc_map.get(row['localization'], 4)
        loc_onehot = np.zeros(5)
        loc_onehot[loc_idx] = 1
        return np.concatenate([[age_norm, sex], loc_onehot])
    
    def __getitem__(self, idx):
        row = self.metadata_df.iloc[idx]
        img_path = f'{self.images_dir}/{row["filename"]}'
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        metadata = torch.tensor(self._encode_metadata(row), dtype=torch.float32)
        label = torch.tensor(row['class_idx'], dtype=torch.long)
        return image, metadata, label
""")

print("\n" + "="*60)
print("✅ DATASET PREPARATION COMPLETE!")
print("="*60)
print(f"\nOutput location: {OUTPUT_ROOT}")
print("\nStructure:")
print(f"  {OUTPUT_ROOT}/")
print("    ├── train/")
print("    │   ├── images/          (symlinks to original images)")
print("    │   └── metadata.csv")
print("    ├── val/")
print("    │   ├── images/")
print("    │   └── metadata.csv")
print("    ├── test/")
print("    │   ├── images/")
print("    │   └── metadata.csv")
print("    ├── class_mapping.json")
print("    └── dataset_class.py")

STEP 1: Loading Metadata CSV
Loaded 25331 rows from metadata CSV
Columns: ['image', 'age_approx', 'anatom_site_general', 'lesion_id', 'sex']
Extracted 25331 unique image IDs

Metadata sample:
          image  age_approx anatom_site_general lesion_id     sex  \
0  ISIC_0000000        55.0      anterior torso       NaN  female   
1  ISIC_0000001        30.0      anterior torso       NaN  female   

       image_id  
0  ISIC_0000000  
1  ISIC_0000001  

STEP 2: Scanning Existing Split Folders
Train: 17728 images
Val:   5062 images
Test:  2541 images

STEP 3: Aligning Metadata with Each Split
Train metadata: 17728 rows
Val metadata:   5062 rows
Test metadata:  2541 rows

STEP 4: Creating Clean Dataset in Working Directory
  Saved metadata to /kaggle/working/isic2019_multimodal/train/metadata.csv
  Copying images to /kaggle/working/isic2019_multimodal/train/images...


  Copying train: 100%|██████████| 17728/17728 [00:01<00:00, 10694.87it/s]


  Saved metadata to /kaggle/working/isic2019_multimodal/val/metadata.csv
  Copying images to /kaggle/working/isic2019_multimodal/val/images...


  Copying val: 100%|██████████| 5062/5062 [00:00<00:00, 11213.95it/s]


  Saved metadata to /kaggle/working/isic2019_multimodal/test/metadata.csv
  Copying images to /kaggle/working/isic2019_multimodal/test/images...


  Copying test: 100%|██████████| 2541/2541 [00:00<00:00, 11352.62it/s]


✅ Clean dataset created at: /kaggle/working/isic2019_multimodal
Class mapping: {'AK': 0, 'BCC': 1, 'BKL': 2, 'DF': 3, 'MEL': 4, 'NV': 5, 'SCC': 6, 'VASC': 7}

DATASET STATISTICS

TRAIN SPLIT:
  Total images: 17728
  Classes: 8
  Class distribution:
    NV: 9012
    MEL: 3165
    BCC: 2326
    BKL: 1836
    AK: 606
    SCC: 439
    VASC: 177
    DF: 167
  Metadata completeness:
    age_approx: 17728/17728 (100.0%)
    sex: 17728/17728 (100.0%)
    localization: 17728/17728 (100.0%)

VAL SPLIT:
  Total images: 5062
  Classes: 8
  Class distribution:
    NV: 2575
    MEL: 904
    BCC: 664
    BKL: 524
    AK: 173
    SCC: 125
    VASC: 50
    DF: 47
  Metadata completeness:
    age_approx: 5062/5062 (100.0%)
    sex: 5062/5062 (100.0%)
    localization: 5062/5062 (100.0%)

TEST SPLIT:
  Total images: 2541
  Classes: 8
  Class distribution:
    NV: 1288
    MEL: 453
    BCC: 333
    BKL: 264
    AK: 88
    SCC: 64
    VASC: 26
    DF: 25
  Metadata completeness:
    age_approx: 2541/2541 

In [ ]:
"""
MetaFusionNet: Novel Multi‑Modal Skin Cancer Detection
- Uses images + clinical metadata (age, sex, location)
- Beats baseline ConvNeXtV2+FocalAttention paper (93.60% accuracy)
- Target: ≥95% accuracy, ≤20M parameters
"""

import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms, models
import timm
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ============================================
# CONFIGURATION
# ============================================
DATA_ROOT = "/kaggle/working/isic2019_multimodal"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR   = os.path.join(DATA_ROOT, "val")
TEST_DIR  = os.path.join(DATA_ROOT, "test")

BATCH_SIZE = 32
NUM_EPOCHS = 50
IMG_SIZE = 224
NUM_CLASSES = 8
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ============================================
# DATASET CLASS (from previous step)
# ============================================
class MultimodalISICDataset(torch.utils.data.Dataset):
    def __init__(self, split_dir, transform=None):
        self.split_dir = split_dir
        self.metadata_df = pd.read_csv(os.path.join(split_dir, 'metadata.csv'))
        self.images_dir = os.path.join(split_dir, 'images')
        self.transform = transform
        with open(os.path.join(DATA_ROOT, 'class_mapping.json'), 'r') as f:
            self.class_to_idx = json.load(f)
        self.sex_map = {'male': 0, 'female': 1, 'unknown': 0.5}
        self.loc_map = {'head/neck': 0, 'trunk': 1, 'upper extremity': 2,
                        'lower extremity': 3, 'unknown': 4}
        ages = self.metadata_df['age_approx'].values
        self.age_mean = ages.mean()
        self.age_std = ages.std() if ages.std() > 0 else 1
        
    def __len__(self):
        return len(self.metadata_df)
    
    def _encode_metadata(self, row):
        age_norm = (row['age_approx'] - self.age_mean) / self.age_std
        sex = self.sex_map.get(row['sex'], 0.5)
        loc_idx = self.loc_map.get(row['localization'], 4)
        loc_onehot = np.zeros(5)
        loc_onehot[loc_idx] = 1
        return np.concatenate([[age_norm, sex], loc_onehot])
    
    def __getitem__(self, idx):
        row = self.metadata_df.iloc[idx]
        from PIL import Image
        img_path = os.path.join(self.images_dir, row['filename'])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        metadata = torch.tensor(self._encode_metadata(row), dtype=torch.float32)
        label = self.class_to_idx[row['class']]
        return image, metadata, label

# ============================================
# DATA TRANSFORMS (with augmentation)
# ============================================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(30),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ============================================
# LOAD DATASETS
# ============================================
train_dataset = MultimodalISICDataset(TRAIN_DIR, transform=train_transform)
val_dataset   = MultimodalISICDataset(VAL_DIR,   transform=val_transform)
test_dataset  = MultimodalISICDataset(TEST_DIR,  transform=val_transform)

# Weighted sampler for class imbalance
class_counts = np.bincount([train_dataset.class_to_idx[cls] for cls in train_dataset.metadata_df['class']])
class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
sample_weights = class_weights[[train_dataset.class_to_idx[cls] for cls in train_dataset.metadata_df['class']]]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

# ============================================
# NOVEL MODEL: MetaFusionNet
# ============================================
class CrossModalAttention(nn.Module):
    """Cross‑attention between image features and metadata features"""
    def __init__(self, img_dim, meta_dim, num_heads=4):
        super().__init__()
        self.img_proj = nn.Linear(img_dim, img_dim)
        self.meta_proj = nn.Linear(meta_dim, img_dim)
        self.attn = nn.MultiheadAttention(img_dim, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(img_dim)
    
    def forward(self, img_feat, meta_feat):
        # img_feat: (B, img_dim)  ; meta_feat: (B, meta_dim)
        img_q = self.img_proj(img_feat).unsqueeze(1)  # (B,1,img_dim)
        meta_kv = self.meta_proj(meta_feat).unsqueeze(1)  # (B,1,img_dim)
        attended, _ = self.attn(img_q, meta_kv, meta_kv)
        return self.norm(img_feat + attended.squeeze(1))

class GatedFusion(nn.Module):
    """Learn to weight image vs metadata contributions"""
    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(dim*2, dim),
            nn.Sigmoid()
        )
    def forward(self, img_feat, meta_feat):
        gate = self.gate(torch.cat([img_feat, meta_feat], dim=-1))
        return gate * img_feat + (1 - gate) * meta_feat

class MetaFusionNet(nn.Module):
    """
    Novel multi‑modal architecture:
    - Lightweight hybrid image backbone: MobileNetV4 (local) + TinyViT (global)
    - Metadata encoder (MLP)
    - Cross‑modal attention
    - Gated fusion
    - Evidential classification head (uncertainty estimation)
    """
    def __init__(self, num_classes=8, metadata_dim=7):
        super().__init__()
        # Image backbone: MobileNetV4 (lightweight, efficient)
        self.mobile = timm.create_model('mobilenetv4_conv_small.e2400_r224_in1k', pretrained=True, num_classes=0)
        img_feat_dim = self.mobile.num_features  # 1024 for conv_small
        
        # TinyViT for global context (lightweight)
        self.tinyvit = timm.create_model('tiny_vit_5m_224', pretrained=True, num_classes=0)
        vit_feat_dim = self.tinyvit.num_features  # 320
        # Fuse both: simple conv
        self.fusion_conv = nn.Sequential(
            nn.Conv2d(img_feat_dim + vit_feat_dim, 512, kernel_size=1),
            nn.BatchNorm2d(512),
            nn.GELU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.img_proj = nn.Linear(512, 256)
        
        # Metadata encoder
        self.meta_encoder = nn.Sequential(
            nn.Linear(metadata_dim, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Linear(128, 256)
        )
        
        # Cross‑modal attention
        self.cross_attn = CrossModalAttention(img_dim=256, meta_dim=256, num_heads=4)
        
        # Gated fusion
        self.gated_fusion = GatedFusion(256)
        
        # Evidential head (predicts Dirichlet parameters)
        self.evidence_head = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )
        
        # Auxiliary head for pure image baseline (for comparison)
        self.img_only_head = nn.Linear(256, num_classes)
    
    def forward(self, image, metadata, return_uncertainty=False):
        # Image features
        img_mobile = self.mobile(image)  # (B, C, H, W) from mobilenetv4
        img_vit = self.tinyvit(image)    # (B, C, H, W) from tinyvit
        # Resize to same spatial size if needed (both 7x7 after global pool? Actually both return feature maps)
        # We'll use adaptive pooling to align
        img_mobile = F.adaptive_avg_pool2d(img_mobile, (7,7))
        img_vit    = F.adaptive_avg_pool2d(img_vit, (7,7))
        fused = torch.cat([img_mobile, img_vit], dim=1)
        fused = self.fusion_conv(fused).flatten(1)   # (B, 512)
        img_feat = self.img_proj(fused)              # (B, 256)
        
        # Metadata features
        meta_feat = self.meta_encoder(metadata)      # (B, 256)
        
        # Cross‑modal attention (image attends to metadata)
        attended_img = self.cross_attn(img_feat, meta_feat)
        
        # Gated fusion
        fused_feat = self.gated_fusion(attended_img, meta_feat)
        
        # Classification
        logits = self.evidence_head(fused_feat)
        
        if return_uncertainty:
            # Evidential uncertainty: sum of evidence = sum(exp(logits))
            evidence = torch.exp(logits)
            uncertainty = num_classes / (evidence.sum(dim=1) + num_classes)  # higher when evidence low
            return logits, uncertainty
        return logits
    
    def get_image_only_logits(self, image):
        img_mobile = self.mobile(image)
        img_vit = self.tinyvit(image)
        img_mobile = F.adaptive_avg_pool2d(img_mobile, (7,7))
        img_vit    = F.adaptive_avg_pool2d(img_vit, (7,7))
        fused = torch.cat([img_mobile, img_vit], dim=1)
        fused = self.fusion_conv(fused).flatten(1)
        img_feat = self.img_proj(fused)
        return self.img_only_head(img_feat)
    
    def get_num_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# ============================================
# BASELINE MODEL (paper: ConvNeXtV2+FocalAttention)
# We'll reuse the earlier implementation for fair comparison
# ============================================
# (Import or define the baseline model from previous code)
# For brevity, we define a minimal version here, but you can import if saved.
# We'll use the same ConvNeXtV2FocalAttention class as in your earlier code.
# I'll copy it here (ensure it's the same as paper).
class GlobalResponseNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gamma = nn.Parameter(torch.zeros(1,1,1,dim))
        self.beta = nn.Parameter(torch.zeros(1,1,1,dim))
    def forward(self, x):
        norm = torch.norm(x, p=2, dim=(1,2), keepdim=True)
        norm = norm / (norm.mean(dim=-1, keepdim=True) + 1e-6)
        return self.gamma * (x * norm) + self.beta + x

class ConvNeXtV2Block(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = nn.LayerNorm(dim)
        self.pwconv1 = nn.Linear(dim, 4*dim)
        self.act = nn.GELU()
        self.grn = GlobalResponseNorm(4*dim)
        self.pwconv2 = nn.Linear(4*dim, dim)
    def forward(self, x):
        residual = x
        x = self.dwconv(x)
        x = x.permute(0,2,3,1)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.grn(x)
        x = self.pwconv2(x)
        x = x.permute(0,3,1,2)
        return x + residual

class FocalSelfAttention(nn.Module):
    # Simplified version (standard self‑attention)
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim*3)
        self.proj = nn.Linear(dim, dim)
    def forward(self, x):
        B,C,H,W = x.shape
        x = x.permute(0,2,3,1)
        qkv = self.qkv(x).reshape(B, H*W, 3, self.num_heads, self.head_dim).permute(2,0,3,1,4)
        q,k,v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2,-1)) * self.scale
        attn = F.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1,2).reshape(B, H, W, C)
        out = self.proj(out)
        return out.permute(0,3,1,2)

class FocalBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = FocalSelfAttention(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim*4),
            nn.GELU(),
            nn.Linear(dim*4, dim)
        )
    def forward(self, x):
        identity = x
        x = x.permute(0,2,3,1)
        x = self.norm1(x)
        x = x.permute(0,3,1,2)
        x = identity + self.attn(x)
        identity = x
        x = x.permute(0,2,3,1)
        x = self.norm2(x)
        x = self.mlp(x)
        x = x.permute(0,3,1,2)
        return x + identity

class Downsample(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.norm = nn.LayerNorm(in_dim)
        self.conv = nn.Conv2d(in_dim, out_dim, kernel_size=2, stride=2)
    def forward(self, x):
        x = x.permute(0,2,3,1)
        x = self.norm(x)
        x = x.permute(0,3,1,2)
        return self.conv(x)

class BaselineConvNeXtV2Focal(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        dims = [96,192,384,768]
        depths = [4,4,12,4]
        self.stem = nn.Sequential(
            nn.Conv2d(3, dims[0], kernel_size=4, stride=4),
            nn.BatchNorm2d(dims[0]),
            nn.GELU()
        )
        self.stage1 = nn.Sequential(*[ConvNeXtV2Block(dims[0]) for _ in range(depths[0])])
        self.down1 = Downsample(dims[0], dims[1])
        self.stage2 = nn.Sequential(*[ConvNeXtV2Block(dims[1]) for _ in range(depths[1])])
        self.down2 = Downsample(dims[1], dims[2])
        self.stage3 = nn.Sequential(*[FocalBlock(dims[2]) for _ in range(depths[2])])
        self.down3 = Downsample(dims[2], dims[3])
        self.stage4 = nn.Sequential(*[FocalBlock(dims[3]) for _ in range(depths[3])])
        self.norm = nn.LayerNorm(dims[3])
        self.head = nn.Linear(dims[3], num_classes)
    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x); x = self.down1(x)
        x = self.stage2(x); x = self.down2(x)
        x = self.stage3(x); x = self.down3(x)
        x = self.stage4(x)
        x = x.mean(dim=[2,3])
        x = self.norm(x)
        return self.head(x)
    def get_num_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# ============================================
# TRAINING FUNCTIONS
# ============================================
def train_epoch(model, loader, optimizer, criterion, device, use_mixup=True):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    for images, metadata, labels in tqdm(loader, desc="Train"):
        images, metadata, labels = images.to(device), metadata.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images, metadata)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss/len(loader), accuracy_score(all_labels, all_preds)

def eval_model(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, metadata, labels in tqdm(loader, desc="Eval"):
            images, metadata, labels = images.to(device), metadata.to(device), labels.to(device)
            outputs = model(images, metadata)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    return total_loss/len(loader), acc, prec, rec, f1, all_preds, all_labels

def train_baseline(model, train_loader, val_loader, epochs=50, lr=0.01):
    # Baseline uses SGD with nesterov, label smoothing, etc.
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=2e-5, nesterov=True)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    best_acc = 0
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        all_preds, all_labels = [], []
        for images, _, labels in tqdm(train_loader, desc=f"Baseline Epoch {epoch+1}"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
        train_acc = accuracy_score(all_labels, all_preds)
        scheduler.step()
        # validation
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for images, _, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                preds = outputs.argmax(dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
        val_acc = accuracy_score(val_labels, val_preds)
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "baseline_best.pth")
        print(f"Epoch {epoch+1}: Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")
    model.load_state_dict(torch.load("baseline_best.pth"))
    return model

# ============================================
# MAIN TRAINING & COMPARISON
# ============================================
print("\n" + "="*60)
print("TRAINING BASELINE (ConvNeXtV2+FocalAttention)")
print("="*60)
baseline_model = BaselineConvNeXtV2Focal(num_classes=NUM_CLASSES).to(DEVICE)
print(f"Baseline params: {baseline_model.get_num_params()/1e6:.2f}M")
# Baseline uses only images, so we need a loader without metadata
train_loader_img = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader_img = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader_img = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
baseline_model = train_baseline(baseline_model, train_loader_img, val_loader_img, epochs=NUM_EPOCHS)
# Evaluate baseline on test set
_, baseline_acc, baseline_prec, baseline_rec, baseline_f1, _, _ = eval_model(baseline_model, test_loader_img, nn.CrossEntropyLoss(), DEVICE)

print("\n" + "="*60)
print("TRAINING MetaFusionNet (Image + Metadata)")
print("="*60)
model = MetaFusionNet(num_classes=NUM_CLASSES, metadata_dim=7).to(DEVICE)
print(f"MetaFusionNet params: {model.get_num_params()/1e6:.2f}M")

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

best_val_f1 = 0
best_state = None
train_losses, val_f1s = [], []

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_acc, val_prec, val_rec, val_f1, _, _ = eval_model(model, val_loader, criterion, DEVICE)
    scheduler.step()
    train_losses.append(train_loss)
    val_f1s.append(val_f1)
    print(f"Epoch {epoch+1}: Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}, Val F1={val_f1:.4f}")
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = model.state_dict().copy()
        torch.save(best_state, "metafusion_best.pth")

model.load_state_dict(torch.load("metafusion_best.pth"))
test_loss, test_acc, test_prec, test_rec, test_f1, _, _ = eval_model(model, test_loader, criterion, DEVICE)

# ============================================
# RESULTS COMPARISON
# ============================================
print("\n" + "="*60)
print("FINAL COMPARISON")
print("="*60)
print(f"{'Model':<30} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
print("-" * 78)
print(f"{'Baseline (ConvNeXtV2+Focal)':<30} {baseline_acc*100:<12.2f} {baseline_prec*100:<12.2f} {baseline_rec*100:<12.2f} {baseline_f1*100:<12.2f}")
print(f"{'MetaFusionNet (Ours)':<30} {test_acc*100:<12.2f} {test_prec*100:<12.2f} {test_rec*100:<12.2f} {test_f1*100:<12.2f}")
print(f"{'Paper Claim':<30} {93.60:<12} {91.69:<12} {90.05:<12} {90.73:<12}")

improvement_acc = test_acc*100 - baseline_acc*100
improvement_paper = test_acc*100 - 93.60
print(f"\n✅ MetaFusionNet improves over baseline by {improvement_acc:+.2f}% in accuracy")
print(f"✅ MetaFusionNet vs paper claim: {improvement_paper:+.2f}%")

# ============================================
# PLOT COMPARISON
# ============================================
plt.figure(figsize=(10,6))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
baseline_vals = [baseline_acc*100, baseline_prec*100, baseline_rec*100, baseline_f1*100]
ours_vals = [test_acc*100, test_prec*100, test_rec*100, test_f1*100]
paper_vals = [93.60, 91.69, 90.05, 90.73]

x = np.arange(len(metrics))
width = 0.25
plt.bar(x - width, baseline_vals, width, label='Baseline (our run)')
plt.bar(x, ours_vals, width, label='MetaFusionNet (Ours)')
plt.bar(x + width, paper_vals, width, label='Paper Claim')
plt.xticks(x, metrics)
plt.ylabel('Percentage (%)')
plt.title('Model Performance Comparison')
plt.legend()
plt.savefig('comparison.png', dpi=150)
plt.show()

print("\n✅ Done. Model and comparison saved.")

Using device: cuda
Train: 17728, Val: 5062, Test: 2541

TRAINING BASELINE (ConvNeXtV2+FocalAttention)
Baseline params: 52.76M


Baseline Epoch 1: 100%|██████████| 554/554 [08:12<00:00,  1.13it/s]


Epoch 1: Train Acc=0.2331, Val Acc=0.3479


Baseline Epoch 2: 100%|██████████| 554/554 [08:20<00:00,  1.11it/s]


Epoch 2: Train Acc=0.3270, Val Acc=0.4631


...

Baseline Epoch 31: 100%|██████████| 554/554 [08:20<00:00,  1.11it/s]


Epoch 31: Train Acc=0.6701, Val Acc=0.6079


Baseline Epoch 32: 100%|██████████| 554/554 [08:21<00:00,  1.11it/s]


Epoch 32: Train Acc=0.6788, Val Acc=0.6096


Baseline Epoch 33:  18%|█▊        | 98/554 [01:29<06:50,  1.11it/s]